In [1]:
import os
from dotenv import load_dotenv

from openai import OpenAI
from qdrant_client import QdrantClient

In [2]:
load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
qdrant_client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "renovation_docs"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-nano"

## Embedding query


In [3]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=EMBED_MODEL,
        input=query,
    )
    return response.data[0].embedding

## Qdrant Retreival

In [9]:
def retrieve_chunks(query: str, top_k: int = 5):
    query_vector = embed_query(query)

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return response.points

## Context

In [5]:
def build_context(points) -> str:
    context_parts = []

    for index, point in enumerate(points, start=1):
        payload = point.payload or {}

        url = payload.get("url", "")
        title = payload.get("title", "")
        text = payload.get("text", "")

        part = (
            f"[Источник {index}]\n"
            f"URL: {url}\n"
            f"Title: {title}\n"
            f"Text:\n{text}"
        )
        context_parts.append(part)

    return "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(context_parts)

## Prompt

In [13]:
SYSTEM_PROMPT = """
Ты — AI-помощник компании по ремонту квартир.

Правила ответа:
1. Отвечай только на основе переданного контекста.
2. Не выдумывай цены, услуги, сроки, адреса, гарантии или условия, если их нет в контексте.
3. Если точного ответа в контексте нет, так и скажи: "Не нашел точной информации в базе знаний".
4. Если в контексте есть несколько релевантных вариантов, коротко перечисли их.
5. Отвечай по-русски.
6. Максимум 5-7 коротких предложений.
7. Отвечай ясно, кратко и по делу.
8. Если в ответе есть цена, укажи единицу измерения, если она есть в контексте.
"""

## Generator

In [14]:
def answer_with_rag(question: str, top_k: int = 5):
    points = retrieve_chunks(question, top_k=top_k)
    context = build_context(points)

    user_prompt = f"""
Вопрос пользователя:
{question}

Контекст:
{context}
"""

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.output_text, points, context

# Tests

In [22]:
answer, points, context = answer_with_rag("Мне нужен ремонт квартиры 40 кв.м. сколько будет примерно стоить?")
print(answer)

Примерная стоимость ремонта квартиры 40 кв.м. составляет от 600 000 рублей и выше, при цене от 15 000₽/м². Точная цена зависит от выбранного типа ремонта и требований к материалам и работам.
